In [1]:
# Run these small tutorials on CPU; select a GPU before importing JAX for larger runs.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import marlax as mx
from marlax.envs import CooperativeGrid, GridState
from marlax.models import QTable
from marlax.methods import QLearning, SARSA, ExpectedSARSA


In [2]:
# 1. Structure is immutable; numerical task parameters are separate.
env = CooperativeGrid(size=5, num_agents=2)
params = env.default_params()
key, reset_key = jax.random.split(jax.random.PRNGKey(0))
state, timestep = env.reset(reset_key, params)
{"spec": env.spec, "parameters": params, "positions": np.asarray(state.positions),
 "observation": np.asarray(timestep.policy_input.obs)}


{'spec': EnvSpec(num_agents=2, num_observations=4375, num_actions=5, observation_schema='coop_grid_joint_index_v1'),
 'parameters': GridParams(time_limit=48, target_reward=1.0, activation_reward=0.2, step_reward=-0.01, wrong_target_reward=-1.0, split_target_reward=-0.4),
 'positions': array([[2, 1],
        [2, 2]], dtype=int32),
 'observation': array([307, 307], dtype=int32)}

In [3]:
# 2. The joint index hides the target until both animals reach the center.
state = GridState(jnp.array([[2, 2], [2, 1]]), jnp.array(0), jnp.array(False), jnp.array(0))
other_target = state._replace(target=jnp.array(5))
assert env.state_id(state) == env.state_id(other_target)
key, step_key = jax.random.split(key)
state, timestep = env.step(step_key, state, jnp.array([0, 2]), params)
assert timestep.metrics["activated"]
{"positions": np.asarray(state.positions), "reward": np.asarray(timestep.reward),
 "revealed_observation": np.asarray(timestep.policy_input.obs)}


{'positions': array([[2, 2],
        [2, 2]], dtype=int32),
 'reward': array([0.19, 0.19], dtype=float32),
 'revealed_observation': array([937, 937], dtype=int32)}

In [4]:
# 3. Both agents move north twice and collect the same selected target.
# Terminal output is the actual final world, not an automatically reset world.
for actions in ([2, 2], [2, 2]):
    key, step_key = jax.random.split(key)
    state, timestep = env.step(step_key, state, jnp.array(actions), params)
assert timestep.terminated and timestep.metrics["collected"]
{"final_positions": np.asarray(state.positions), "reward": np.asarray(timestep.reward),
 "terminated": bool(timestep.terminated), "truncated": bool(timestep.truncated)}


{'final_positions': array([[2, 4],
        [2, 4]], dtype=int32),
 'reward': array([0.99, 0.99], dtype=float32),
 'terminated': True,
 'truncated': False}

In [5]:
# 4. Reset is explicit. A time limit is an external truncation, not task failure.
short_params = params._replace(time_limit=1)
state = GridState(jnp.zeros((2, 2), dtype=jnp.int32), jnp.array(0), jnp.array(False), jnp.array(0))
key, step_key, reset_key = jax.random.split(key, 3)
final_state, timestep = env.step(step_key, state, jnp.array([0, 0]), short_params)
assert timestep.truncated and not timestep.terminated
reset_state, reset_timestep = env.reset(reset_key, short_params)
{"final_step": int(final_state.steps), "reset_step": int(reset_state.steps),
 "reset_reward": np.asarray(reset_timestep.reward)}


{'final_step': 1,
 'reset_step': 0,
 'reset_reward': array([0., 0.], dtype=float32)}

In [6]:
# 5. vmap adds parallel worlds without changing environment physics.
keys = jax.random.split(jax.random.PRNGKey(4), 8)
states, timesteps = jax.vmap(env.reset, in_axes=(0, None))(keys, params)
{"positions": states.positions.shape, "observations": timesteps.policy_input.obs.shape,
 "action_masks": timesteps.policy_input.action_mask.shape}


{'positions': (8, 2, 2), 'observations': (8, 2), 'action_masks': (8, 2, 5)}